### Import Libraries

In [32]:
! pip install transformers
! pip install datasets

In [106]:
!pip install gradio -q

In [107]:
import pandas as pd 
import gradio as gr
from transformers import pipeline
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,AutoModelForMaskedLM
from datasets import Dataset
import re
from sklearn.metrics import accuracy_score, f1_score
import torch
import warnings
warnings.filterwarnings('ignore')

### Read & Explore Data

In [80]:
df = pd.read_csv('Final_Data.csv')
df.head()

,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,talbat


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   review_description  40045 non-null  object
 1   rating              40046 non-null  object
 2   company             40046 non-null  object
dtypes: object(3)
memory usage: 938.7+ KB


In [82]:
df.drop("company", axis=1, inplace=True)

In [83]:
df.duplicated().sum()

np.int64(941)

In [84]:
df.drop_duplicates(inplace=True)

In [85]:
df.isna().sum()

,0
review_description,1
rating,0


In [86]:
df.dropna(inplace=True)

In [87]:
df['rating'] = df['rating'].map({'positive': 1, 'negative': 0})

In [88]:
df['rating'].value_counts()

,count
rating,
1.0,23211
0.0,13975


### Text Preprocessing 

In [89]:
df.rename(columns={'review_description': 'text', 'rating': 'labels'}, inplace=True)

In [90]:
def preprocess_for_transformer(text):
    text = re.sub(r'<.*?>', '', text)          
    text = re.sub(r'http\S+|www\S+', '', text) 
    text = re.sub(r'@\w+|#\w+', '', text)      
    text = re.sub(r'(.)\\1{3,}', r'\1\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [91]:
df['text'] = df['text'].apply(preprocess_for_transformer)

In [92]:
df = df.dropna(subset=['text', 'labels'])

In [93]:
df['labels'] = df['labels'].astype(int)

### MARBERTv2 Transformer

In [94]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

In [97]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [98]:
model_name = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,           # positive / negative
    problem_type="single_label_classification"
)

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

In [99]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     padding="max_length",
                     truncation=True,
                     max_length=128)

In [100]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/23798 [00:00<?, ? examples/s]

Map:   0%|          | 0/5950 [00:00<?, ? examples/s]

Map:   0%|          | 0/7438 [00:00<?, ? examples/s]

In [101]:
training_args = TrainingArguments(
    output_dir="./marbert_sentiment",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=762,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

In [102]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

In [103]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [104]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.286182,0.262335,0.905210,0.905270
2,0.227155,0.267516,0.908235,0.908107
3,0.177133,0.337425,0.905378,0.905590
4,0.122488,0.438937,0.896975,0.897366
5,0.095380,0.483076,0.898824,0.898906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=7440, training_loss=0.19048848716161584, metrics={'train_runtime': 1498.9934, 'train_samples_per_second': 79.38, 'train_steps_per_second': 4.963, 'total_flos': 7826896119321600.0, 'train_loss': 0.19048848716161584, 'epoch': 5.0})

In [105]:
results = trainer.evaluate(test_dataset)
print(results)

{'eval_loss': 0.2658767104148865, 'eval_accuracy': 0.9105942457649906, 'eval_f1': 0.9104273747005066, 'eval_runtime': 14.8411, 'eval_samples_per_second': 501.176, 'eval_steps_per_second': 15.7, 'epoch': 5.0}


### Save weights

In [116]:
SAVE_PT         = "marbert_sentiment.pt"     
SAVE_TOKENIZER  = "marbert_tokenizer"         

In [117]:
torch.save(model.state_dict(), SAVE_PT)

In [118]:
tokenizer.save_pretrained(SAVE_TOKENIZER)

('marbert_tokenizer/tokenizer_config.json', 'marbert_tokenizer/tokenizer.json')